In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


BASE_DIR = Path.cwd().parent
BASE_DIR

DATA_DIR = BASE_DIR/"data"/"processed"
MODELS_DIR = BASE_DIR /"models"
parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

In [4]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 


In [5]:
train = pd.read_parquet(DATA_DIR/'train_filtered_ca1.parquet')
test  = pd.read_parquet(DATA_DIR/'test_filtered_ca1.parquet')



In [50]:
### Analysing the experiment results calculated
import json 

results_dir = BASE_DIR /'results/Experiments'

lgbm_direct = results_dir/'lgbm_direct_1.0yr.json'
lgbm_recursive= results_dir/'lgbm_recursive_1.0yr.json'

lgbm_direct_2 = results_dir/'lgbm_direct_2.0yr.json'
lgbm_recursive_2 = results_dir/'lgbm_recursive_2.0yr.json'

expt_1 = [lgbm_recursive,lgbm_direct]

In [ ]:

def compare_expts(recursive_file,direct_file):
    """
    Extracts only monthly holding cost entries from the FVA list.
    
    Parameters:
        data (list or dict): The raw experiment output.
        
    Returns:
        pd.DataFrame: A DataFrame containing filtered holding cost FVA entries.
    """
    with open(recursive_file,'r') as f1:
        rec_data = json.load(f1)
    with open(direct_file,'r') as f2:
        direct_data = json.load(f2)

    to_rename = ['lgbm','fva_model_vs_naive']
    merge_on = ['window_id','train_start','train_end','metric','seasonal_naive',
                   'moving_average','fva_moving_average_vs_naive']

    rec_fva = pd.DataFrame(rec_data[0].get('fva')).rename(columns= {**{col : f'{col}_recursive' for col in to_rename}})
    direc_fva = pd.DataFrame(direct_data[0].get('fva')).rename(columns= {**{col : f'{col}_direct' for col in to_rename}})

    selected_cols = ['window_id','train_start','train_end','model','MAE','BIAS%','wrmsse']

    rec_metric = pd.DataFrame(rec_data[0].get('metrics'))[selected_cols]
    direc_metric = pd.DataFrame(direct_data[0].get('metrics'))[selected_cols]

    
    # pivot to table 
    pivot_table = lambda df : df.pivot(
                index=["window_id", "train_start", "train_end"],
                columns="model",
            ).stack(level=0, future_stack=True).reset_index()

    rec_metric_table = pivot_table(rec_metric).rename(columns={'lgbm':'lgbm_recursive','level_3':'metrics'})
    direc_metric_table = pivot_table(rec_metric).rename(columns={'lgbm':'lgbm_direct','level_3':'metrics'})

    merge_metric_on = ['window_id','train_start','train_end','metrics','seasonal_naive','moving_average']

    metric_merge = rec_metric_table.merge(direc_metric_table,on=merge_metric_on,how='left')

    # direc_metric = pd.DataFrame(direc_fva[0].get('metric'))


    # print(rec_fva.columns,direc_fva.columns)
    merge = rec_fva.merge(direc_fva,on=merge_on,how='left').reset_index(drop=True)

    return merge, metric_merge
    # return pd.DataFrame(metric),pd.DataFrame(fva_records) 



In [116]:
fva_comparison, metric = compare_expts(lgbm_recursive,lgbm_direct)

ValueError: If using all scalar values, you must pass an index

In [100]:
metric

model,window_id,train_start,train_end,metrics,lgbm_recursive,moving_average,seasonal_naive,lgbm_direct
0,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,MAE,1.835817,1.197533,1.267262,1.042466
1,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,BIAS%,-0.345006,9.532267,-0.717280,-1.811691
2,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,wrmsse,0.906473,1.005821,1.201814,0.913915
3,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,MAE-DEPT,108.559230,174.926984,105.571429,94.562237
4,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,MAE-CAT,51.777802,362.029630,33.666667,64.828360
...,...,...,...,...,...,...,...,...
115,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,MAE-DEPT,126.287441,88.965079,65.857143,89.972343
116,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,MAE-CAT,285.818374,102.548148,111.666667,162.253057
117,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,monthly_holding_cost_mae,2091.499349,2222.378068,2112.589733,1893.495501
118,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,monthly_holding_cost_classical,1627.536732,1601.256094,1599.752261,1569.368301


In [101]:
# aggregate metrics
# 1. Define the model columns you want to aggregate
model_cols = ['lgbm_recursive', 'moving_average', 'seasonal_naive', 'lgbm_direct']

# 2. Group by 'metrics' and apply aggregations across all model columns
agg_metrics = metric.groupby('metrics')[model_cols].agg(['mean', 'median', 'std'])

agg_metrics

model                          lgbm_recursive                           \
                                         mean       median         std   
metrics                                                                  
BIAS%                                2.502898     2.292658    6.549726   
MAE                                  2.039391     2.048617    0.102713   
MAE-CAT                            256.406357   285.818374  139.798731   
MAE-DEPT                           144.237269   134.886805   49.433638   
monthly_holding_cost_classical    1595.060642  1608.557435   66.660682   
monthly_holding_cost_mae          1988.623211  1948.367214  133.797889   
monthly_holding_cost_rmse         2131.430864  2077.672718  135.455335   
wrmsse                               0.865069     0.863952    0.052874   

model                          moving_average                           \
                                         mean       median         std   
metrics                                                                  
BIAS%                                2.427934     2.325526    6.610428   
MAE                                  1.246350     1.225663    0.051749   
MAE-CAT                            264.793580   275.103704  131.514755   
MAE-DEPT                           154.632593   127.619048   50.035589   
monthly_holding_cost_classical    1587.652092  1574.290669   46.288084   
monthly_holding_cost_mae          2151.374804  2195.175659  119.463812   
monthly_holding_cost_rmse         2292.928500  2325.790493  120.054333   
wrmsse                               0.919914     0.910257    0.061345   

model                          seasonal_naive                           \
                                         mean       median         std   
metrics                                                                  
BIAS%                                1.187552     1.717401    6.768065   
MAE                                  1.405214     1.432619    0.071852   
MAE-CAT                            260.288889   302.333333  149.556480   
MAE-DEPT                           143.019048   130.142857   64.561988   
monthly_holding_cost_classical    1581.424925  1592.789748   62.547115   
monthly_holding_cost_mae          2108.191433  2120.042867  125.175826   
monthly_holding_cost_rmse         2325.994135  2340.619246  138.176701   
wrmsse                               1.185407     1.182145    0.045196   

model                           lgbm_direct                           
                                       mean       median         std  
metrics                                                               
BIAS%                             -1.257955    -1.099751    6.050727  
MAE                                1.149125     1.146889    0.055322  
MAE-CAT                          246.519008   212.052582  130.195433  
MAE-DEPT                         140.408086   124.407412   49.376329  
monthly_holding_cost_classical  1565.979180  1576.161781   66.241339  
monthly_holding_cost_mae        1927.713942  1906.265381  122.918159  
monthly_holding_cost_rmse       2068.546974  2038.849522  129.925186  
wrmsse                             0.864037     0.854348    0.052177

In [102]:
fva_comparison

,window_id,train_start,train_end,metric,lgbm_recursive,moving_average,seasonal_naive,fva_moving_average_vs_naive,fva_model_vs_naive_recursive,lgbm_direct,fva_model_vs_naive_direct
0,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,MAE,1.835817,1.197533,1.267262,5.50,-44.86,1.042466,17.74
1,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,wrmsse,0.906473,1.005821,1.201814,16.31,24.57,0.913915,23.96
2,1,2014-11-10T00:00:00.000,2015-11-10T00:00:00.000,MAE,1.882558,1.216985,1.339762,9.16,-40.51,1.083982,19.09
3,1,2014-11-10T00:00:00.000,2015-11-10T00:00:00.000,wrmsse,0.863952,0.962356,1.210809,20.52,28.65,0.875128,27.72
4,1,2014-11-10T00:00:00.000,2015-11-10T00:00:00.000,monthly_holding_cost_mae,1854.318042,2205.118217,1887.933367,-16.80,1.78,1794.326711,4.96
...,...,...,...,...,...,...,...,...,...,...,...
67,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,MAE,2.226427,1.296722,1.357500,4.48,-64.01,1.135597,16.35
68,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,wrmsse,0.866377,1.002596,1.182145,15.19,26.71,0.899959,23.87
69,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,monthly_holding_cost_mae,2091.499349,2222.378068,2112.589733,-5.20,1.00,1893.495501,10.37
70,14,2013-11-11T00:00:00.000,2014-11-11T00:00:00.000,monthly_holding_cost_classical,1627.536732,1601.256094,1599.752261,-0.09,-1.74,1569.368301,1.90


In [105]:
# 1. Define the model columns you want to aggregate
fva_cols = ['fva_moving_average_vs_naive','fva_model_vs_naive_recursive','fva_model_vs_naive_direct']

# 2. Group by 'metrics' and apply aggregations across all model columns
fva_agg_metrics = fva_comparison.groupby('metric')[fva_cols].agg(['mean', 'median', 'std'])

fva_agg_metrics

fva_moving_average_vs_naive                    \
                                                      mean  median       std   
metric                                                                         
MAE                                              11.176000  10.490  4.124522   
monthly_holding_cost_classical                   -0.450714  -0.475  2.211600   
monthly_holding_cost_mae                         -2.245714  -2.825  6.286697   
monthly_holding_cost_rmse                         1.230714   1.020  5.798198   
wrmsse                                           22.413333  23.790  3.923906   

                               fva_model_vs_naive_recursive                    \
                                                       mean  median       std   
metric                                                                          
MAE                                              -45.280000 -43.360  6.867289   
monthly_holding_cost_classical                    -0.862857  -0.785  1.209007   
monthly_holding_cost_mae                           5.632857   6.090  4.017867   
monthly_holding_cost_rmse                          8.312857   8.935  3.786633   
wrmsse                                            27.055333  27.050  2.629554   

                               fva_model_vs_naive_direct                    
                                                    mean  median       std  
metric                                                                      
MAE                                            18.196667  17.780  1.691836  
monthly_holding_cost_classical                  0.981429   0.870  1.153509  
monthly_holding_cost_mae                        8.511429   7.965  3.682215  
monthly_holding_cost_rmse                      11.017857  10.925  3.512116  
wrmsse                                         27.133333  26.490  2.852943

In [106]:
fva_cols = [
    'fva_moving_average_vs_naive',
    'fva_model_vs_naive_recursive',
    'fva_model_vs_naive_direct'
]

# 2. Map full column names to clean model labels
fva_model_map = {
    'fva_moving_average_vs_naive': 'moving_average',
    'fva_model_vs_naive_recursive': 'recursive',
    'fva_model_vs_naive_direct': 'direct'
}

# Find the column with the maximum FVA value for each row
winning_cols = fva_comparison[fva_cols].idxmax(axis=1)

# Map the column names to simple model names ('direct', 'recursive', 'moving_average')
fva_comparison['winning_model'] = winning_cols.map(fva_model_map)

# --- STEP 2: Aggregate Wins Per Metric Across All Windows ---

# Count how many times each model won for each metric
metric_win_counts = (
    fva_comparison
    .groupby(['metric', 'winning_model'])
    .size()
    .unstack(fill_value=0)
)

# Identify the overall dominant model per metric across all windows
overall_metric_winner = (
    fva_comparison
    .groupby('metric')['winning_model']
    .agg(lambda x: x.mode()[0] if not x.empty else None)
    .reset_index(name='overall_winning_model')
)

# --- DISPLAY RESULTS ---
print("--- Per-Window Winners ---")
print(fva_comparison[['window_id', 'metric', 'winning_model'] + fva_cols])

print("\n--- Total Win Counts per Metric ---")
print(metric_win_counts)

print("\n--- Overall Dominant Model per Metric ---")
print(overall_metric_winner)

--- Per-Window Winners ---
    window_id                          metric winning_model  \
0           0                             MAE        direct   
1           0                          wrmsse     recursive   
2           1                             MAE        direct   
3           1                          wrmsse     recursive   
4           1        monthly_holding_cost_mae        direct   
..        ...                             ...           ...   
67         14                             MAE        direct   
68         14                          wrmsse     recursive   
69         14        monthly_holding_cost_mae        direct   
70         14  monthly_holding_cost_classical        direct   
71         14       monthly_holding_cost_rmse        direct   

    fva_moving_average_vs_naive  fva_model_vs_naive_recursive  \
0                          5.50                        -44.86   
1                         16.31                         24.57   
2                    

In [107]:
overall_metric_winner

,metric,overall_winning_model
0,MAE,direct
1,monthly_holding_cost_classical,direct
2,monthly_holding_cost_mae,direct
3,monthly_holding_cost_rmse,direct
4,wrmsse,recursive


In [ ]:
import pandas as pd
import numpy as np

def Calculate_fva_and_savings(df):
    """
    Calculates percentage wins over Seasonal Naive and Moving Average,
    identifies the winning model per window, and computes dollar savings
    for holding cost metrics.
    """
    data = df.copy()
    
    # 1. Define Model and Baseline columns
    models = ['lgbm_recursive', 'lgbm_direct', 'moving_average']
    
    # --- PERCENTAGE WINS OVER SEASONAL NAIVE ---
    # Formula: ((Naive - Model) / Naive) * 100
    data['pct_win_recursive_vs_naive'] = ((data['seasonal_naive'] - data['lgbm_recursive']) / data['seasonal_naive']) * 100
    data['pct_win_direct_vs_naive'] = ((data['seasonal_naive'] - data['lgbm_direct']) / data['seasonal_naive']) * 100
    data['pct_win_ma_vs_naive'] = ((data['seasonal_naive'] - data['moving_average']) / data['seasonal_naive']) * 100

    # --- PERCENTAGE WINS OVER MOVING AVERAGE ---
    # Formula: ((MA - Model) / MA) * 100
    data['pct_win_recursive_vs_ma'] = ((data['moving_average'] - data['lgbm_recursive']) / data['moving_average']) * 100
    data['pct_win_direct_vs_ma'] = ((data['moving_average'] - data['lgbm_direct']) / data['moving_average']) * 100

    # --- IDENTIFY WINNING MODEL (Lowest value is best for error/cost metrics) ---
    # Map model column names to clean labels
    model_labels = {
        'lgbm_recursive': 'recursive',
        'lgbm_direct': 'direct',
        'moving_average': 'moving_average',
        'seasonal_naive': 'seasonal_naive'
    }
    
    all_competitors = ['lgbm_recursive', 'lgbm_direct', 'moving_average', 'seasonal_naive']
    winning_cols = data[all_competitors].idxmin(axis=1)
    data['winning_model'] = winning_cols.map(model_labels)
    
    # Extract the winning value for each row
    data['winning_value'] = data[all_competitors].min(axis=1)

    # --- DOLLAR SAVINGS FOR COST METRICS ---
    # Identify holding cost rows (where metric starts with 'monthly_holding_cost')
    is_cost_metric = data['metrics'].str.startswith('monthly_holding_cost', na=False)
    
    # Dollars saved by the overall winner vs baselines
    data['dollars_saved_vs_naive'] = np.where(
        is_cost_metric, 
        data['seasonal_naive'] - data['winning_value'], 
        0.0
    )
    
    data['dollars_saved_vs_ma'] = np.where(
        is_cost_metric, 
        data['moving_average'] - data['winning_value'], 
        0.0
    )

    return data


# ==========================================
# EXAMPLE USAGE & SAMPLE DATA
# ==========================================

# Creating sample input matching your DataFrame structure
sample_data = pd.DataFrame({
    'train_start': ['2013-11-11', '2013-11-11'],
    'train_end': ['2014-11-11', '2014-11-11'],
    'metrics': ['monthly_holding_cost_mae', 'monthly_holding_cost_classical'],
    'lgbm_recursive': [2091.49, 1627.53],
    'moving_average': [2222.37, 1601.25],
    'seasonal_naive': [2112.58, 1599.75],
    'lgbm_direct': [1893.49, 1569.36]
})

# Run processing function
processed_fva = calculate_fva_and_savings(sample_data)

# Select relevant columns for clear display
display_cols = [
    'metrics', 
    'winning_model', 
    'pct_win_direct_vs_naive', 
    'pct_win_direct_vs_ma', 
    'dollars_saved_vs_naive', 
    'dollars_saved_vs_ma'
]

print("--- FVA & COST SAVINGS SUMMARY ---")
print(processed_fva[display_cols].to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import html

# ============================================================
# SETTINGS
# ============================================================

TITLE = "FVA Model Comparison — Recursive vs Direct"
SUBTITLE = "1-Year  | Backtest Summary"

OUTPUT = "fva_model_summary.html"


# ============================================================
# DATA
# ============================================================
fva_table = fva_comparison

df = fva_table.copy()


# ============================================================
# ACCURACY SUMMARY
# ============================================================

accuracy = df[df["metric"].isin(["MAE", "wrmsse"])].copy()

accuracy_summary = (
    accuracy.groupby("metric")
    .agg(
        Recursive_Mean_FVA=("fva_model_vs_naive_recursive", "mean"),
        Direct_Mean_FVA=("fva_model_vs_naive_direct", "mean"),

        Recursive_Median_FVA=("fva_model_vs_naive_recursive", "median"),
        Direct_Median_FVA=("fva_model_vs_naive_direct", "median"),

        Recursive_Win_Rate=("fva_model_vs_naive_recursive",
                            lambda x: (x > 0).mean() * 100),

        Direct_Win_Rate=("fva_model_vs_naive_direct",
                         lambda x: (x > 0).mean() * 100),

        Windows=("window_id", "nunique")
    )
    .round(2)
)


# ============================================================
# WINDOW SUMMARY
# ============================================================

window_summary = accuracy[
    [
        "window_id",
        "train_start",
        "train_end",
        "metric",
        "fva_model_vs_naive_recursive",
        "fva_model_vs_naive_direct"
    ]
].copy()

window_summary["Winner"] = window_summary.apply(
    lambda x:
        "Direct"
        if x["fva_model_vs_naive_direct"]
        > x["fva_model_vs_naive_recursive"]
        else "Recursive",
    axis=1
)

window_summary.columns = [
    "Window",
    "Train Start",
    "Train End",
    "Metric",
    "Recursive FVA (%)",
    "Direct FVA (%)",
    "Winner"
]


# ============================================================
# COST SUMMARY
# ============================================================

cost_metrics = [
    "monthly_holding_cost_mae",
    "monthly_holding_cost_classical",
    "monthly_holding_cost_rmse"
]

cost = df[df["metric"].isin(cost_metrics)]

cost_summary = (
    cost.groupby("metric")
    .agg(
        Recursive_Mean_FVA=("fva_model_vs_naive_recursive", "mean"),
        Direct_Mean_FVA=("fva_model_vs_naive_direct", "mean"),

        Recursive_Win_Rate=("fva_model_vs_naive_recursive",
                            lambda x: (x > 0).mean() * 100),

        Direct_Win_Rate=("fva_model_vs_naive_direct",
                         lambda x: (x > 0).mean() * 100)
    )
    .round(2)
)


# ============================================================
# OVERALL MODEL SCORE
# ============================================================

recursive = accuracy["fva_model_vs_naive_recursive"]
direct = accuracy["fva_model_vs_naive_direct"]

recursive_wins = (recursive > direct).sum()
direct_wins = (direct > recursive).sum()

overall = {
    "Recursive Mean FVA": recursive.mean(),
    "Direct Mean FVA": direct.mean(),
    "Recursive Wins": recursive_wins,
    "Direct Wins": direct_wins,
}


# ============================================================
# HTML TABLE HELPER
# ============================================================

def make_table(data):
    return data.to_html(
        classes="data-table",
        border=0,
        float_format=lambda x: f"{x:.2f}"
    )


# ============================================================
# HTML
# ============================================================

html_page = f"""
<!DOCTYPE html>

<html>
<head>

<meta charset="UTF-8">

<title>{html.escape(TITLE)}</title>

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f5f6f8;
    color: #222;
    margin: 0;
}}

.container {{
    max-width: 1400px;
    margin: auto;
    padding: 40px;
}}

h1 {{
    margin-bottom: 5px;
}}

.subtitle {{
    color: #666;
    margin-bottom: 35px;
}}

.section {{
    background: white;
    padding: 25px;
    margin-bottom: 25px;
    border-radius: 10px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.06);
}}

.cards {{
    display: flex;
    gap: 15px;
    flex-wrap: wrap;
}}

.card {{
    background: #f1f3f5;
    padding: 20px;
    border-radius: 8px;
    min-width: 190px;
}}

.value {{
    font-size: 28px;
    font-weight: bold;
}}

.label {{
    color: #666;
    margin-top: 5px;
}}

.data-table {{
    width: 100%;
    border-collapse: collapse;
    margin-top: 15px;
}}

.data-table th {{
    background: #343a40;
    color: white;
    padding: 10px;
}}

.data-table td {{
    padding: 9px;
    border-bottom: 1px solid #ddd;
    text-align: center;
}}

.data-table tr:hover {{
    background: #f5f5f5;
}}

select {{
    padding: 7px;
}}

.note {{
    color: #666;
    font-size: 14px;
}}

</style>

<script>

function filterMetric() {{

    const selected =
        document.getElementById("metric").value;

    const rows =
        document.querySelectorAll("#window-table tbody tr");

    rows.forEach(row => {{

        const metric = row.cells[4].innerText;

        row.style.display =
            selected === "All" || metric === selected
            ? ""
            : "none";

    }});
}}

</script>

</head>


<body>

<div class="container">

<h1>{html.escape(TITLE)}</h1>

<div class="subtitle">
{html.escape(SUBTITLE)}
</div>


<!-- ====================================================== -->
<!-- OVERALL -->
<!-- ====================================================== -->

<div class="section">

<h2>Overall Model Comparison</h2>

<div class="cards">

<div class="card">
<div class="value">{overall["Recursive Mean FVA"]:.2f}%</div>
<div class="label">Recursive Mean FVA</div>
</div>

<div class="card">
<div class="value">{overall["Direct Mean FVA"]:.2f}%</div>
<div class="label">Direct Mean FVA</div>
</div>

<div class="card">
<div class="value">{overall["Recursive Wins"]}</div>
<div class="label">Recursive Wins</div>
</div>

<div class="card">
<div class="value">{overall["Direct Wins"]}</div>
<div class="label">Direct Wins</div>
</div>

</div>

</div>


<!-- ====================================================== -->
<!-- ACCURACY -->
<!-- ====================================================== -->

<div class="section">

<h2>Accuracy Summary</h2>

<p class="note">
Positive FVA means improvement over the seasonal-naive benchmark.
Higher FVA is therefore better.
</p>

{make_table(accuracy_summary)}

</div>


<!-- ====================================================== -->
<!-- WINDOW -->
<!-- ====================================================== -->

<div class="section">

<h2>Window-by-Window Performance</h2>

<select id="metric" onchange="filterMetric()">

<option value="All">All</option>
<option value="MAE">MAE</option>
<option value="wrmsse">WRMSSE</option>

</select>

<div id="window-table">

{make_table(window_summary)}

</div>

</div>


<!-- ====================================================== -->
<!-- COST -->
<!-- ====================================================== -->

<div class="section">

<h2>Inventory Cost Impact</h2>

<p class="note">
Positive FVA means the model has lower cost than the
seasonal-naive benchmark.
</p>

{make_table(cost_summary)}

</div>


<!-- ====================================================== -->
<!-- INTERPRETATION -->
<!-- ====================================================== -->

<div class="section">

<h2>Model Selection Checklist</h2>

<ul>

<li>Compare mean FVA across all windows.</li>

<li>Check median FVA to avoid conclusions driven by a few windows.</li>

<li>Check win rate for consistency across windows.</li>

<li>Compare MAE and WRMSSE separately.</li>

<li>Check whether accuracy improvements translate into
lower inventory cost.</li>

<li>Inspect the window-by-window results before selecting
the final deployment model.</li>

</ul>

</div>


</div>

</body>
</html>
"""


# ============================================================
# SAVE
# ============================================================

Path(OUTPUT).write_text(
    html_page,
    encoding="utf-8"
)

print(f"Saved: {OUTPUT}")

Saved: fva_model_summary.html


In [71]:
from IPython.display import display, HTML

display(HTML("fva_model_summary.html"))

,Recursive_Mean_FVA,Direct_Mean_FVA,Recursive_Median_FVA,Direct_Median_FVA,Recursive_Win_Rate,Direct_Win_Rate,Windows
metric,,,,,,,
MAE,-45.28,18.20,-43.36,17.78,0.00,100.00,15
wrmsse,27.06,27.13,27.05,26.49,100.00,100.00,15
,Window,Train Start,Train End,Metric,Recursive FVA (%),Direct FVA (%),Winner
0,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,MAE,-44.86,17.74,Direct
1,0,2014-12-08T00:00:00.000,2015-12-08T00:00:00.000,wrmsse,24.57,23.96,Recursive
2,1,2014-11-10T00:00:00.000,2015-11-10T00:00:00.000,MAE,-40.51,19.09,Direct
3,1,2014-11-10T00:00:00.000,2015-11-10T00:00:00.000,wrmsse,28.65,27.72,Recursive
7,2,2014-10-13T00:00:00.000,2015-10-13T00:00:00.000,MAE,-42.74,22.51,Direct
8,2,2014-10-13T00:00:00.000,2015-10-13T00:00:00.000,wrmsse,32.59,33.47,Direct


In [69]:
Deployment_dir = MODELS_DIR /'deployments/deployment_1788773717'



forecasts = pd.read_parquet(Deployment_dir/'forecasts.parquet')
inventory_costs = pd.read_parquet(Deployment_dir/'inventory_costs.parquet')
inventory_policy = pd.read_parquet(Deployment_dir/'inventory_policy.parquet')


In [5]:
# comparison with 
forecast_comparison  = pd.read_parquet(Deployment_dir/'forecast_comparison.parquet')
inventory_policy_comparison = pd.read_parquet(Deployment_dir/'inventory_policy_comparison.parquet')
inventory_cost_comparison = pd.read_parquet(Deployment_dir/'inventory_cost_comparison.parquet')


In [6]:
forecast_comparison

,item_id,dept_id,cat_id,date,sales_pred,model,item_std,q10,q50,q90,q95,q97,real_sales
0,FOODS_1_001,FOODS_1,FOODS,2016-01-06,0.708841,lgbm,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_002,FOODS_1,FOODS,2016-01-06,0.312749,lgbm,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_003,FOODS_1,FOODS,2016-01-06,0.463815,lgbm,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_005,FOODS_1,FOODS,2016-01-06,0.965672,lgbm,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_006,FOODS_1,FOODS,2016-01-06,2.335216,lgbm,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101131,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,2016-01-29,0.000000,seasonal_naive,0.472368,0.0,0.0,0.605364,0.776977,0.888427,0.0
101132,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,2016-01-30,0.000000,seasonal_naive,0.472368,0.0,0.0,0.605364,0.776977,0.888427,1.0
101133,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,2016-01-31,0.000000,seasonal_naive,0.472368,0.0,0.0,0.605364,0.776977,0.888427,1.0
101134,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,2016-02-01,0.000000,seasonal_naive,0.472368,0.0,0.0,0.605364,0.776977,0.888427,0.0


In [7]:
inventory_policy_comparison

,item_id,rmse_tau,mae_tau,n_obs,forecast_tau,safety_stock_rmse,safety_stock_mae,order_up_to_rmse,order_up_to_mae,raw_demand_std,safety_stock_classical,order_up_to_classical,review_date,protection_end_date,model
0,FOODS_1_001,2.828687,2.505070,18,9.261779,4.653191,4.120841,13.914970,13.382620,1.077433,5.878311,15.140090,2016-01-06,2016-01-16,lgbm
1,FOODS_1_002,1.873357,1.535078,18,5.114355,3.081673,2.525203,8.196028,7.639559,0.689638,3.762557,8.876913,2016-01-06,2016-01-16,lgbm
2,FOODS_1_003,4.760875,4.228223,18,8.630641,7.831639,6.955426,16.462280,15.586068,1.140808,6.224075,14.854717,2016-01-06,2016-01-16,lgbm
3,FOODS_1_005,19.795138,15.113437,18,16.810702,32.563002,24.861603,49.373704,41.672305,1.631536,8.901414,25.712115,2016-01-06,2016-01-16,lgbm
4,FOODS_1_006,5.385523,4.233985,18,17.175556,8.859185,6.964906,26.034741,24.140461,1.728662,9.431317,26.606873,2016-01-06,2016-01-16,lgbm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10831,HOUSEHOLD_2_509,2.505549,1.944444,18,5.000000,4.121629,3.198611,9.121629,8.198611,0.900655,4.913835,9.913835,2016-01-20,2016-01-30,seasonal_naive
10832,HOUSEHOLD_2_511,3.480102,3.111111,18,0.000000,5.724768,5.117778,5.724768,5.117778,1.401590,7.646863,7.646863,2016-01-20,2016-01-30,seasonal_naive
10833,HOUSEHOLD_2_512,2.357023,1.777778,18,4.000000,3.877302,2.924444,7.877302,6.924444,1.110556,6.059026,10.059026,2016-01-20,2016-01-30,seasonal_naive
10834,HOUSEHOLD_2_514,2.236068,2.000000,18,3.000000,3.678332,3.290000,6.678332,6.290000,0.499433,2.724831,5.724831,2016-01-20,2016-01-30,seasonal_naive


In [8]:
inventory_cost_comparison

,item_id,rmse_tau,mae_tau,n_obs,forecast_tau,safety_stock_rmse,safety_stock_mae,order_up_to_rmse,order_up_to_mae,raw_demand_std,...,protection_end_date,cycle_demand_R,cycle_stock,avg_on_hand_rmse,monthly_holding_cost_rmse,avg_on_hand_mae,monthly_holding_cost_mae,avg_on_hand_classical,monthly_holding_cost_classical,model
0,FOODS_1_001,2.828687,2.505070,18,9.261779,4.653191,4.120841,13.914970,13.382620,1.077433,...,2016-01-16,6.031775,3.015888,7.669078,1.073671,7.136728,0.999142,8.894199,1.245188,lgbm
1,FOODS_1_002,1.873357,1.535078,18,5.114355,3.081673,2.525203,8.196028,7.639559,0.689638,...,2016-01-16,3.550888,1.775444,4.857117,0.679996,4.300647,0.602091,5.538001,0.775320,lgbm
2,FOODS_1_003,4.760875,4.228223,18,8.630641,7.831639,6.955426,16.462280,15.586068,1.140808,...,2016-01-16,6.227014,3.113507,10.945146,1.532320,10.068933,1.409651,9.337582,1.307262,lgbm
3,FOODS_1_005,19.795138,15.113437,18,16.810702,32.563002,24.861603,49.373704,41.672305,1.631536,...,2016-01-16,12.642655,6.321327,38.884330,5.443806,31.182931,4.365610,15.222741,2.131184,lgbm
4,FOODS_1_006,5.385523,4.233985,18,17.175556,8.859185,6.964906,26.034741,24.140461,1.728662,...,2016-01-16,10.360256,5.180128,14.039313,1.965504,12.145034,1.700305,14.611445,2.045602,lgbm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10831,HOUSEHOLD_2_509,2.505549,1.944444,18,5.000000,4.121629,3.198611,9.121629,8.198611,0.900655,...,2016-01-30,2.000000,1.000000,5.121629,0.717028,4.198611,0.587806,5.913835,0.827937,seasonal_naive
10832,HOUSEHOLD_2_511,3.480102,3.111111,18,0.000000,5.724768,5.117778,5.724768,5.117778,1.401590,...,2016-01-30,0.000000,0.000000,5.724768,0.801468,5.117778,0.716489,7.646863,1.070561,seasonal_naive
10833,HOUSEHOLD_2_512,2.357023,1.777778,18,4.000000,3.877302,2.924444,7.877302,6.924444,1.110556,...,2016-01-30,2.000000,1.000000,4.877302,0.682822,3.924444,0.549422,7.059026,0.988264,seasonal_naive
10834,HOUSEHOLD_2_514,2.236068,2.000000,18,3.000000,3.678332,3.290000,6.678332,6.290000,0.499433,...,2016-01-30,3.000000,1.500000,5.178332,0.724966,4.790000,0.670600,4.224831,0.591476,seasonal_naive


In [9]:
from src.utils_visuals import * 


inventory_cost_model = inventory_cost_comparison[inventory_cost_comparison['model']=='lgbm']



In [10]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from config import PIPELINE_CONFIG
from src.utils_visuals import (
    plot_item_forecast_and_inventory,
    plot_inventory_policy_bars,
)

# Set this explicitly to the deployment you want to show.
artifact_dir = Path("models/deployments/deployment_1788773717")

forecasts = pd.read_parquet(Deployment_dir / "forecast_comparison.parquet")
policies = pd.read_parquet(Deployment_dir / "inventory_policy_comparison.parquet")
costs = pd.read_parquet(Deployment_dir / "inventory_cost_comparison.parquet")
test_actuals = pd.read_parquet(Deployment_dir / "test_actuals.parquet")

# Compatibility for artifacts created before actual_sales was standardized.
if "actual_sales" not in forecasts.columns:
    actuals = test_actuals.rename(columns={"sales": "actual_sales"})
    forecasts = (
        forecasts.drop(columns=["real_sales"], errors="ignore")
        .merge(
            actuals[["item_id", "dept_id", "cat_id", "date", "actual_sales"]],
            on=["item_id", "dept_id", "cat_id", "date"],
            how="left",
        )
    )

item_id = "FOODS_1_001"  # choose any deployed SKU
models = ["lgbm", "moving_average", "seasonal_naive"]

# Historical context + known test actuals
train = pd.read_parquet(PIPELINE_CONFIG["train_data_path"])
history = (
    train.loc[train["item_id"] == item_id, ["date", "sales"]]
    .sort_values("date")
    .tail(90)
)

item_forecasts = forecasts.query("item_id == @item_id").copy()
test_sales = (
    item_forecasts[["date", "actual_sales"]]
    .drop_duplicates()
    .rename(columns={"actual_sales": "sales"})
)

raw_sales = (
    pd.concat([history, test_sales], ignore_index=True)
    .drop_duplicates("date", keep="last")
    .sort_values("date")
)

forecast_lines = {
    model: item_forecasts.loc[
        item_forecasts["model"] == model, ["date", "sales_pred"]
    ]
    for model in models
}

# Optional ML quantile band; works whether quantiles were configured or not.
lgb = item_forecasts.query("model == 'lgbm'").copy()
has_quantiles = (
    {"q10", "q90"}.issubset(lgb.columns)
    and lgb["q10"].notna().any()
    and lgb["q90"].notna().any()
)

p10 = lgb[["date", "q10"]].rename(columns={"q10": "p10"}) if has_quantiles else None
p90 = lgb[["date", "q90"]].rename(columns={"q90": "p90"}) if has_quantiles else None
p95 = (
    lgb[["date", "q95"]].rename(columns={"q95": "p95"})
    if has_quantiles and "q95" in lgb.columns and lgb["q95"].notna().any()
    else None
)

fig_forecast = plot_item_forecast_and_inventory(
    raw_sales=raw_sales,
    forecasted_demand=forecast_lines,
    p10=p10,
    p90=p90,
    p95=p95,
    inventory_values=None,  # Inventory is shown separately: it is tau-day, not daily demand.
    item_id=item_id,
)
fig_forecast.show()

In [11]:
item_policy = (
    policies.query("item_id == @item_id")
    .sort_values("review_date")
    .groupby("model", as_index=False)
    .first()
)

item_cost = (
    costs.query("item_id == @item_id")
    .sort_values("review_date")
    .groupby("model", as_index=False)
    .first()
)

policy_levels = {
    row["model"]: {
        "Forecast τ demand": row["forecast_tau"],
        "Safety stock (RMSE)": row["safety_stock_rmse"],
        "Order-up-to (RMSE)": row["order_up_to_rmse"],
    }
    for _, row in item_policy.iterrows()
}

fig_policy = plot_inventory_policy_bars(
    policy_levels,
    metric_name="Units",
    item_id=item_id,
)
fig_policy.show()

holding_cost = {
    row["model"]: row["monthly_holding_cost_rmse"]
    for _, row in item_cost.iterrows()
}

fig_holding = plot_inventory_policy_bars(
    holding_cost,
    metric_name="Estimated holding cost per review period",
    item_id=item_id,
)
fig_holding.show()

In [12]:
portfolio_cost = (
    costs.groupby("model", as_index=False)["monthly_holding_cost_rmse"]
    .sum()
    .rename(columns={"monthly_holding_cost_rmse": "estimated_holding_cost"})
)

fig_portfolio = px.bar(
    portfolio_cost,
    x="model",
    y="estimated_holding_cost",
    text_auto=".2f",
    title="Estimated Policy-Implied Holding Cost Across Test Horizon",
    labels={
        "model": "Forecast model",
        "estimated_holding_cost": "Estimated holding cost",
    },
)
fig_portfolio.show()